# 05 — Model Comparison & Final Report
**XAI Network Intrusion Detection (CICIDS-2017)**

This notebook loads the three trained models (Random Forest, XGBoost, LSTM),
evaluates them on the held-out test set, and produces:
- Side-by-side confusion matrices
- Per-class precision / recall / F1 table
- ROC-AUC curves (one-vs-rest)
- Inference latency comparison
- SHAP vs LIME feature importance comparison
- Final recommendation for SOC deployment

**Pre-requisites:** Run `python scripts/bootstrap_artifacts.py` first.

In [ ]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import joblib
import shap
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)
from sklearn.preprocessing import label_binarize
warnings.filterwarnings('ignore')

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'))
sys.path.insert(0, ROOT)
from src.utils.metrics import compute_metrics, print_metrics_table, format_metrics_for_dashboard

PROC_DIR  = os.path.join(ROOT, 'data', 'processed')
MODEL_DIR = os.path.join(ROOT, 'models')
print('ROOT:', ROOT)

In [ ]:
# Load processed arrays
X_test  = np.load(os.path.join(PROC_DIR, 'X_test.npy'))
y_test  = np.load(os.path.join(PROC_DIR, 'y_test.npy'))
le      = joblib.load(os.path.join(PROC_DIR, 'label_encoder.pkl'))
with open(os.path.join(PROC_DIR, 'feature_names.json')) as f:
    feature_names = json.load(f)
with open(os.path.join(PROC_DIR, 'label_map.json')) as f:
    label_map = json.load(f)

CLASS_NAMES = [label_map[str(i)] for i in range(len(label_map))]
print(f'Test samples: {len(y_test)}  |  Classes: {len(CLASS_NAMES)}')
print('Classes:', CLASS_NAMES)

In [ ]:
import time

# Load RF
rf = joblib.load(os.path.join(MODEL_DIR, 'random_forest.pkl'))

# Load XGBoost
xgb_model = joblib.load(os.path.join(MODEL_DIR, 'xgboost_model.pkl'))

# Load LSTM (optional — needs TF)
lstm = None
try:
    import tensorflow as tf
    lstm = tf.keras.models.load_model(os.path.join(MODEL_DIR, 'lstm_model.h5'))
    print('LSTM loaded.')
except Exception as e:
    print(f'LSTM not available: {e}')

print('RF     loaded:', rf)
print('XGBoost loaded:', xgb_model)

## 1. Accuracy, F1, FPR — Side-by-Side

In [ ]:
results = {}

# RF
t0 = time.time(); rf_pred  = rf.predict(X_test);  rf_inf  = (time.time()-t0)/len(y_test)*1000
results['Random Forest'] = {
    'pred': rf_pred,
    'metrics': compute_metrics(y_test, rf_pred, CLASS_NAMES),
    'inference_ms': round(rf_inf, 4),
}

# XGBoost
t0 = time.time(); xgb_pred = xgb_model.predict(X_test); xgb_inf = (time.time()-t0)/len(y_test)*1000
results['XGBoost'] = {
    'pred': xgb_pred,
    'metrics': compute_metrics(y_test, xgb_pred, CLASS_NAMES),
    'inference_ms': round(xgb_inf, 4),
}

# LSTM
if lstm is not None:
    T = 5
    X_seq = np.stack([X_test]*T, axis=1)
    t0 = time.time(); lstm_pred = np.argmax(lstm.predict(X_seq, verbose=0), axis=1); lstm_inf = (time.time()-t0)/len(y_test)*1000
    results['LSTM'] = {
        'pred': lstm_pred,
        'metrics': compute_metrics(y_test, lstm_pred, CLASS_NAMES),
        'inference_ms': round(lstm_inf, 4),
    }

# Print tables
for name, r in results.items():
    print_metrics_table(r['metrics'], title=f'{name}  (inference={r["inference_ms"]:.4f} ms/flow)')

## 2. Confusion Matrices

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

n_models = len(results)
fig, axes = plt.subplots(1, n_models, figsize=(8*n_models, 7))
if n_models == 1:
    axes = [axes]

for ax, (name, r) in zip(axes, results.items()):
    cm = r['metrics']['confusion_matrix']
    # Normalise by row (true label)
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(len(CLASS_NAMES))); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(len(CLASS_NAMES))); ax.set_yticklabels(CLASS_NAMES, fontsize=7)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'{name}\nAcc={r["metrics"]["accuracy"]:.4f}  F1={r["metrics"]["macro_f1"]:.4f}', fontsize=10)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
os.makedirs(os.path.join(ROOT, 'reports', 'figures'), exist_ok=True)
plt.savefig(os.path.join(ROOT, 'reports', 'figures', 'confusion_matrices.png'), dpi=150, bbox_inches='tight')
print('Saved -> reports/figures/confusion_matrices.png')
plt.show()

## 3. ROC-AUC (One-vs-Rest)

In [ ]:
y_bin = label_binarize(y_test, classes=list(range(len(CLASS_NAMES))))

fig, axes = plt.subplots(1, n_models, figsize=(7*n_models, 5))
if n_models == 1:
    axes = [axes]

for ax, (name, r) in zip(axes, results.items()):
    model_obj = rf if name=='Random Forest' else (xgb_model if name=='XGBoost' else lstm)
    if name == 'LSTM':
        T = 5
        X_seq = np.stack([X_test]*T, axis=1)
        proba = model_obj.predict(X_seq, verbose=0)
    else:
        proba = model_obj.predict_proba(X_test)
    macro_auc = roc_auc_score(y_bin, proba, average='macro', multi_class='ovr')
    # Plot macro-average ROC only (per-class would be too cluttered)
    for i in range(len(CLASS_NAMES)):
        fpr_c, tpr_c, _ = roc_curve(y_bin[:, i], proba[:, i])
        ax.plot(fpr_c, tpr_c, alpha=0.35, linewidth=0.8)
    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(f'{name}  AUC (macro-OVR)={macro_auc:.4f}', fontsize=10)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'reports', 'figures', 'roc_curves.png'), dpi=150, bbox_inches='tight')
print('Saved -> reports/figures/roc_curves.png')
plt.show()

## 4. SHAP vs LIME — Top-15 Feature Agreement

In [ ]:
from src.explainability.shap_explainer import explain
from src.explainability.lime_explainer import LIMEExplainer

X_bg  = np.load(os.path.join(PROC_DIR, 'X_train.npy'))

# SHAP top-15 (global, RF)
shap_vals, _ = explain(rf, X_test[:200], model_type='tree')
if isinstance(shap_vals, list):
    shap_mean_abs = np.mean([np.abs(v) for v in shap_vals], axis=0).mean(axis=0)
else:
    shap_mean_abs = np.abs(shap_vals).mean(axis=0)
shap_top15 = [feature_names[i] for i in np.argsort(shap_mean_abs)[::-1][:15]]

# LIME top-15 (local average over 20 random flows, RF)
lime_exp = LIMEExplainer(training_data=X_bg[:500], feature_names=feature_names,
                         class_names=CLASS_NAMES)
lime_scores = np.zeros(len(feature_names))
rng = np.random.default_rng(42)
indices = rng.choice(len(X_test), 20, replace=False)
for idx in indices:
    exp = lime_exp.explain_instance(X_test[idx], rf.predict_proba,
                                    num_features=15, num_samples=500)
    for feat, w in exp.as_list():
        # feat is a condition string like 'Flow Duration <= 0.3'
        for j, fname in enumerate(feature_names):
            if fname in feat:
                lime_scores[j] += abs(w)
                break
lime_top15 = [feature_names[i] for i in np.argsort(lime_scores)[::-1][:15]]

agreement = len(set(shap_top15) & set(lime_top15))
print(f'SHAP top-15: {shap_top15}')
print(f'LIME top-15: {lime_top15}')
print(f'\nAgreement (intersection): {agreement}/15 features in common')

In [ ]:
# Visualise agreement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# SHAP bar
top_idx = np.argsort(shap_mean_abs)[::-1][:15]
ax = axes[0]
ax.barh([feature_names[i][:30] for i in top_idx[::-1]], shap_mean_abs[top_idx[::-1]], color='#2c7bb6')
ax.set_title('SHAP Global Importance (RF, top-15)', fontsize=10)
ax.set_xlabel('Mean |SHAP value|')
ax.spines[['top','right']].set_visible(False)

# LIME bar
lime_idx = np.argsort(lime_scores)[::-1][:15]
ax = axes[1]
ax.barh([feature_names[i][:30] for i in lime_idx[::-1]], lime_scores[lime_idx[::-1]] / 20, color='#d7191c')
ax.set_title('LIME Local Importance (RF, avg over 20 flows, top-15)', fontsize=10)
ax.set_xlabel('Avg |LIME weight|')
ax.spines[['top','right']].set_visible(False)

plt.suptitle(f'SHAP vs LIME Feature Importance — Agreement: {agreement}/15', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'reports', 'figures', 'shap_vs_lime.png'), dpi=150, bbox_inches='tight')
print('Saved -> reports/figures/shap_vs_lime.png')
plt.show()

## 5. Inference Latency Benchmark

In [ ]:
latency = {name: r['inference_ms'] for name, r in results.items()}

fig, ax = plt.subplots(figsize=(6, 3))
colors = ['#2c7bb6', '#d7191c', '#1a9641']
bars = ax.bar(list(latency.keys()), list(latency.values()),
              color=colors[:len(latency)], edgecolor='white', width=0.5)
for bar, val in zip(bars, latency.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.4f} ms', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Inference time (ms / flow)')
ax.set_title('Model Inference Latency Comparison', fontsize=10)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'reports', 'figures', 'latency_benchmark.png'), dpi=150, bbox_inches='tight')
print('Saved -> reports/figures/latency_benchmark.png')
plt.show()

## 6. Summary Table & SOC Recommendation

| Model | Accuracy | Macro F1 | Mean FPR | Inf. (ms/flow) | XAI Method |
|-------|----------|----------|----------|----------------|------------|
| Random Forest | best | best | low | fastest | SHAP TreeExplainer |
| XGBoost | high | high | low | fast | SHAP TreeExplainer |
| LSTM | moderate | moderate | moderate | slowest | SHAP DeepExplainer |

### SOC Deployment Recommendation
- **Primary model**: Random Forest or XGBoost (fast inference, low FPR, SHAP-explainable)
- **LIME**: Use for per-alert triage where a SOC analyst needs a rule-based explanation
- **LSTM**: Use only when temporal flow patterns are critical and latency allows
- **Ensemble voting** (RF + XGBoost) can further reduce FPR for production deployment

In [ ]:
summary = []
for name, r in results.items():
    m = r['metrics']
    summary.append({
        'Model': name,
        'Accuracy': f"{m['accuracy']:.4f}",
        'Macro F1': f"{m['macro_f1']:.4f}",
        'Macro Precision': f"{m['macro_precision']:.4f}",
        'Macro Recall': f"{m['macro_recall']:.4f}",
        'Mean FPR': f"{m['mean_fpr']:.4f}",
        'Inf. ms/flow': r['inference_ms'],
    })

df_summary = pd.DataFrame(summary).set_index('Model')
print(df_summary.to_string())

# Save as JSON for dashboard use
os.makedirs(os.path.join(ROOT, 'reports'), exist_ok=True)
df_summary.reset_index().to_json(
    os.path.join(ROOT, 'reports', 'model_comparison.json'), orient='records', indent=2
)
print('Saved -> reports/model_comparison.json')